# Chains (LangChain v1.2)

**LCEL(LangChain Expression Language)**을 사용해서 모든 구성 요소가 `Runnable` 인터페이스로 통합되어 파이프라인(`|`)으로 연결될 수 있다.

```python
chain = prompt | model | output_parser  # 기본 구조
```

**구성 요소 업데이트 (v1.2 기준)**
1. **PromptTemplate**  
   - `Runnable`로 변환되어 LCEL 파이프라인에 직접 통합  
   ```python
   prompt = ChatPromptTemplate.from_template("...")
   ```

2. **LLM/ChatModel**  
   - `ChatOpenAI`, `ChatAnthropic` 등이 `Runnable` 구현  
   ```python
   model = ChatOpenAI(model="gpt-4o")
   ```

3. **Memory**  
   - `RunnableWithMessageHistory`로 통합 관리 (또는 LangGraph Persistence 사용)
   ```python
   chain_with_memory = RunnableWithMessageHistory(
       base_chain,
       get_session_history
   )
   ```

4. **Output Parsers**  
   - `StrOutputParser()`, `JsonOutputParser()` 등이 `Runnable`로 작동  
   ```python
   output_parser = JsonOutputParser()
   ```

5. **Tools**  
   - `@tool` 데코레이터로 생성 후 `RunnableLambda`로 변환  
   ```python
   @tool
   def search(query: str) -> str: ...
   ```

**체인 유형별 구현**


1. Simple Chain  

    ```python
    chain = prompt | model | output_parser
    response = chain.invoke({"input": "..."})
    ```

2. Sequential Chain  

    ```python
    chain = (
        {"step1_output": prompt1 | model1}  # 첫 번째 체인 결과 매핑
        | prompt2
        | model2
    )
    ```

3. Conditional Chain
    - `RunnableBranch` 사용

    ```python
    branch = RunnableBranch(
        (lambda x: x["topic"] == "math", math_chain),
        (lambda x: x["topic"] == "history", history_chain),
        default_chain
    )
    ```

4. Memory Chain  

    ```python
    memory_chain = RunnableWithMessageHistory(
        core_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history"
    )
    ```


**🚨 v1.2 주요 변경점**

- **Legacy Chain 클래스 완전 폐기**: `LLMChain`, `SequentialChain` 등은 `langchain-classic`으로 이동되거나 삭제됨 → `Runnable` (LCEL)로 통합
- **에이전트 통합**: `create_agent` (LangGraph 기반)가 표준

In [1]:
# %pip install -Uqqq langchain langchain-openai langchain-community

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')

### Simple Chain

In [4]:
from langchain_core.prompts import PromptTemplate    # prompt chain 구성
from langchain.chat_models import init_chat_model    # 모델 chain 구성 래퍼
from langchain_core.output_parsers import StrOutputParser # 답변 문자형 변환

prompt = PromptTemplate.from_template('{city}의 특산물은 무엇입니까?')
llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

chain = prompt | llm | output_parser
print(chain.invoke('강원도'))
# 프롬프트 템플릿 변수가 2개 이상일 경우 dict형으로 전달
# print(chain.invoke(input={'city':'강원도',...})) 
print()
print(chain.invoke(input={'city':'강원도'})) 

강원도의 대표적인 특산물은 다음과 같습니다.

- **감자**: 평창·강릉·홍천 등에서 많이 생산되며, 강원도를 대표하는 농산물입니다.
- **옥수수**: 특히 홍천과 강릉의 찰옥수수가 유명합니다.
- **고랭지 배추**: 평창·태백·정선 지역의 고랭지 배추가 대표적입니다.
- **메밀**: 봉평 메밀과 메밀국수·메밀전병이 유명합니다.
- **한우**: 횡성한우가 대표적인 축산 특산물입니다.
- **황태**: 인제 용대리 황태가 유명합니다.
- **오징어와 명태**: 강릉·속초·동해 등 동해안 지역의 대표 수산물입니다.
- **더덕·곤드레·산나물**: 정선, 평창, 양구 등 산간 지역에서 많이 생산됩니다.
- **잣**: 홍천 잣이 잘 알려져 있습니다.
- **초당두부**: 강릉 초당 지역의 전통 음식으로 유명합니다.

강원도의 대표적인 특산물은 다음과 같습니다.

- **감자**: 강원도 전역, 특히 평창·강릉 지역
- **옥수수**: 홍천·정선·강릉 등
- **메밀**: 평창 봉평 메밀
- **횡성 한우**
- **인제 황태**
- **속초 오징어·젓갈**
- **강릉 초당두부**
- **양구 곰취**
- **정선 곤드레**
- **춘천 닭갈비·막국수**: 지역을 대표하는 향토 음식

이 밖에도 **더덕, 산나물, 잣, 송이버섯** 등 청정 산지에서 생산되는 임산물도 유명합니다.


In [7]:
prompt1 = PromptTemplate.from_template('다음 내용을 한글로 번역하세요. {eng_text}')
prompt2 = PromptTemplate.from_template('다음 내용을 요약하세요. {kor_text}')
llm = init_chat_model('gpt-5.6-luna')
output_parser = StrOutputParser()

# 번역 체인
chain1 = prompt1 | llm | output_parser
eng_text = """
One limitation of LLMs is their lack of contextual information (e.g., access to some specific documents or emails). You can combat this by giving LLMs access to the specific external data.
For this, you first need to load the external data with a document loader. LangChain provides a variety of loaders for different types of documents ranging from PDFs and emails to websites and YouTube videos.
"""
print(chain1.invoke(eng_text))

chain2 = prompt2 | llm | output_parser

kor_text ="""
LLM의 한 가지 한계는 특정 문서나 이메일과 같은 맥락 정보를 갖고 있지 않다는 점입니다. 이를 해결하려면 LLM이 특정 외부 데이터에 접근할 수 있도록 해야 합니다.
이를 위해서는 먼저 문서 로더를 사용해 외부 데이터를 불러와야 합니다. LangChain은 PDF, 이메일부터 웹사이트, 유튜브 영상에 이르기까지 다양한 유형의 문서를 위한 여러 종류의 로더를 제공합니다.
"""

print(chain2.invoke(kor_text)) 

LLM의 한 가지 한계는 맥락 정보가 부족하다는 점입니다. 예를 들어 특정 문서나 이메일에 접근할 수 없습니다. 이러한 한계를 극복하려면 LLM이 특정 외부 데이터에 접근할 수 있도록 해야 합니다.

이를 위해 먼저 문서 로더(document loader)를 사용하여 외부 데이터를 불러와야 합니다. LangChain은 PDF와 이메일부터 웹사이트 및 YouTube 동영상에 이르기까지 다양한 유형의 문서를 지원하는 여러 로더를 제공합니다.
LLM은 특정 문서나 이메일 같은 외부 맥락 정보가 부족하므로, 필요한 데이터를 제공하려면 외부 데이터에 접근하게 해야 합니다. 이를 위해 문서 로더를 사용하며, LangChain은 PDF, 이메일, 웹사이트, 유튜브 등 다양한 자료를 불러올 수 있는 로더를 제공합니다.


In [6]:
# Sequential Chain
chain = chain1 | chain2
print(chain.invoke({'eng_text': eng_text}))

LLM은 특정 문서나 이메일 등 외부 맥락 정보에 접근하지 못하는 한계가 있습니다. 이를 해결하려면 외부 데이터를 LLM에 연결해야 하며, 먼저 LangChain의 문서 로더를 이용해 PDF, 이메일, 웹사이트, YouTube 동영상 등 다양한 자료를 불러올 수 있습니다.


### Conditional Chain

In [ ]:
from langchain_core.runnables import RunnableBranch

llm = init_chat_model('gpt-5.6-luna')

math_prompt = PromptTemplate.from_template('다음 문제를 풀어주세요. 단계적인 풀이를 수식(LaTex)과 함께 작성해주세요.{question}')
math_chain = math_prompt | llm | output_parser
default_prompt = PromptTemplate.from_template('당신은 친절하고, 감성적이며 공감능력이 좋은 챗봇입니다. 다음 질문에 답변해주세요. {question}')
default_chain = default_prompt | llm | output_parser

# math_chain 선택 함수
def is_math_question(input_dict: dict) -> bool:
    question: str = input_dict.get('question', '') # 입력받은 dict에서 question 키의 값을 추출(없으면 빈 문자열)
    return '계산' in question or 'calc' in question

branch_chain = RunnableBranch(
    (is_math_question,math_chain),
    default_chain
)

print(branch_chain.invoke({'question': '125 * 3 + 50 계산해줘.'}))

단계적으로 계산하면:

\[
125 \times 3 = 375
\]

\[
375 + 50 = 425
\]

따라서 정답은

\[
\boxed{425}
\]


In [10]:
print(branch_chain.invoke({'question': '나 오늘 우울해. 빵? 밥?'}))

오늘은 **따뜻한 밥** 먹자. 국이나 찌개처럼 속 편하고 든든한 걸로. 우울한 날엔 거창하게 잘 먹으려 하지 않아도, 한 숟갈씩 몸을 챙겨주는 게 도움이 될 수 있어.  

근데 빵이 더 당기면 **빵도 완전 괜찮아**—좋아하는 빵에 따뜻한 우유나 차 곁들이자. 오늘은 네 마음이 끌리는 쪽으로 가도 돼. 🫶


### Memory Chain

`RunnableWithMessageHistory`를 사용하여 대화내역을 기억하는 chain을 생성한다.

In [ ]:
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from pydantic import BaseModel, Field
from typing import List

# 사용자별 세션 대화내역을 기록하는 클래스
class InMemoryHistory(BaseChatMessageHistory, BaseModel):
    messages: List[BaseMessage] = Field(default_factory=list) # 인스턴스마다 독립적인 messages list를 구성

    def add_messages(self, messages: list[BaseMessage]) -> None:
        self.messages.extend(messages)
    def clear(self) -> None:
        self.messages = []

store = {} # {session_id: 히스토리 객체(InMemoryHistory)} 저장소

# 세션 ID로 히스토리 객체를 반환
def get_by_session_id(sesstion_id: str) -> BaseChatMessageHistory:
    if sesstion_id not in store:
        store[sesstion_id] = InMemoryHistory() # 기존 대화내역이 없으면 히스토리 객체 생성해서 store에 추가
    return store[sesstion_id] # 해당 세션의 히스토리 객체 반환

history1 = get_by_session_id('1')
history1.add_messages([AIMessage(content='반갑습니다. sungmin님!')])
history1.add_messages([HumanMessage(content='그래 나 min이야 반갑다!')])
print(f"{history1 = }")

history2 = get_by_session_id('2')
print(f"{history2 = }")

history1 = InMemoryHistory(messages=[AIMessage(content='반갑습니다. sungmin님!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='그래 나 min이야 반갑다!', additional_kwargs={}, response_metadata={})])
history2 = InMemoryHistory(messages=[])


### 대화 히스토리를 자동으로 누적하는 Memory Chain (RunnableWithMessageHistory)

In [12]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableWithMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ('system', '당신은 {domain} 분야의 전문가 챗봇입니다.'),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{question}')
])

llm = init_chat_model('gpt-5.6-luna')

chain = prompt | llm

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_by_session_id,
    input_messages_key='question',
    history_messages_key='history'
)
chain_with_history.invoke({
    'domain': 'math',
    'question': '민수는 강아지를 3마리 키우고 있습니다.'
}, config={
    'configurable': {
        'session_id': '100'
    }
})


AIMessage(content='그렇군요. 민수는 강아지 3마리를 키우고 있네요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 38, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 26, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-EH3TpVKPvPfbo71vJhSadFIz7pLNh', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a03d3a-d136-7683-9773-506da759c1d2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 56, 'total_tokens': 94, 'input_token_details': {'audio': 0, 'cache_read': 0, 'cache_creation': 0}, 'output_token_details': {'audio': 0, 'reasoning': 26}})